# 20. Embedding timeout sweep

## 실험 목적

본 실험(notebook 05)에서 MS는 6x6 이상에서 embedding에 실패했습니다. 원인 후보가 셋인데 관측만으로는 구분되지 않습니다.

| 관측 | 원인 |
|---|---|
| timeout/tries를 늘렸더니 성공 | **(a) search budget** |
| Pegasus 실패, Zephyr 성공 | **(b) topology / connectivity** |
| 충분한 budget + Zephyr에서도 실패 | **(c) MS formulation 자체** |

embedding 탐색은 확률적이므로 seed 하나로 판정하지 않고 여러 seed의 **성공률**로 봅니다.

embedding 가능 여부는 QUBO의 **그래프 구조에만** 의존하고 penalty 값과는 무관합니다. 따라서 본 실험의 기본 설정으로 QUBO를 한 번만 만들어 씁니다.

이 notebook은 **tries를 고정하고 timeout을 훑습니다.** tries는 notebook 15에서 따로 다룹니다.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import load_config, resolve_path

config = load_config(PROJECT_ROOT / "config" / "experiment_config.yaml")
DATA_DIR = resolve_path(config, "data_dir")
RAW_DIR = resolve_path(config, "raw_dir")
PROCESSED_DIR = resolve_path(config, "processed_dir")
FIGURE_DIR = resolve_path(config, "figure_dir")
STUDY = config["embedding_study"]

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("설정 로드 완료")


설정 로드 완료


In [2]:
import requests

NTFY_API_KEY = "cflp-formulation-260910"

def notify(message):
    requests.post(
        f"https://ntfy.sh/{NTFY_API_KEY}",
        data=message.encode("utf-8")
    )

## 설정

- `QA_DRYRUN` — True면 이상적 topology 그래프, False면 실제 QPU의 working graph를 대상으로 한다. QPU는 결함 큐빗이 제외되어 있어 이상적 그래프보다 불리하고, 접속한 solver의 topology 하나만 쓸 수 있다.
- 나머지 값은 `config/experiment_config.yaml`의 `embedding_study` 섹션에서 관리한다. 여기서 덮어쓸 수도 있다.

In [3]:
QA_DRYRUN = False

TARGETS = [
    ("4x4", "MS"),
    ("6x6", "MS"),
    ("8x8", "MS"),
    ("15x15", "MS"),
]

TOPOLOGIES = [tuple(item) for item in STUDY["topologies"]]
TIMEOUTS = list(STUDY["timeouts"])
SEEDS = list(STUDY["seeds"])
TRIES = int(STUDY["timeout_sweep_tries"])

print(f"모드       : {'이상적 topology 그래프' if QA_DRYRUN else '실제 QPU working graph'}")
print(f"대상       : {TARGETS}")
print(f"topology   : {TOPOLOGIES}")
print(f"timeout    : {TIMEOUTS}")
print(f"seed       : {SEEDS}")
print(f"tries(고정): {TRIES}")

모드       : 실제 QPU working graph
대상       : [('4x4', 'MS'), ('6x6', 'MS'), ('8x8', 'MS'), ('15x15', 'MS')]
topology   : [('pegasus', 16), ('zephyr', 12)]
timeout    : [300, 600, 1200, 2400]
seed       : [2024, 2025, 2026, 2027, 2028]
tries(고정): 5


## 실행 시간 추정

**실패하는 경우는 timeout을 전부 소진합니다.** 따라서 최악의 경우 소요 시간은 아래와 같습니다. 큰 instance에서는 수 시간이 걸릴 수 있으므로 반드시 먼저 확인하십시오.

시간이 부족하면 `TARGETS`를 줄이거나 `TIMEOUTS`의 큰 값을 빼십시오.

In [4]:
from src import embedding_study as ES

budget = ES.estimate_budget(TARGETS, TOPOLOGIES, TIMEOUTS, SEEDS)
print(f"총 실행 횟수      : {budget['runs']:.0f}")
print(f"최악의 경우(전부 실패): {budget['worst_case_minutes']:.0f}분 "
      f"({budget['worst_case_hours']:.1f}시간)")
print("성공하면 훨씬 짧게 끝납니다.")

총 실행 횟수      : 160
최악의 경우(전부 실패): 3000분 (50.0시간)
성공하면 훨씬 짧게 끝납니다.


## 사용 가능한 solver 확인 (online 모드일 때만)

`QA_DRYRUN=False`이면 지정한 topology를 가진 solver를 골라 접속합니다. **Zephyr는 Advantage2 계열이므로 계정에 해당 solver가 없으면 사용할 수 없습니다.** 아래에서 먼저 확인하십시오.

dry-run이면 이 셀은 건너뜁니다.

In [5]:
from src import embedding_study as ES

if QA_DRYRUN:
    print("dry-run 모드입니다. 이상적 topology 그래프를 사용합니다.")
else:
    try:
        display(ES.list_available_solvers())
    except Exception as error:
        print("solver 목록을 가져오지 못했습니다:", error)

,solver,topology,topology_shape,qubits,couplers
0,Advantage_system4;graph_id=01d07086e1,pegasus,[16],5627,80558
1,Advantage_system6;graph_id=01dae5a273,pegasus,[16],5612,80176
2,Advantage2_system1;graph_id=010c30e58c,zephyr,"[12, 4]",4577,83028


## topology 비교 기준

두 topology의 규모와 연결도를 먼저 확인합니다. Zephyr가 노드도 많고 평균 차수도 높으므로, **Zephyr에서만 성공한다면 그것은 connectivity 덕분**이라고 해석할 수 있습니다.

In [6]:
import dwave_networkx as dnx

rows = []
for name, size in TOPOLOGIES:
    builder = {"pegasus": dnx.pegasus_graph, "zephyr": dnx.zephyr_graph}[name]
    graph = builder(size)
    rows.append({
        "topology": f"{name}{size}",
        "nodes": graph.number_of_nodes(),
        "edges": graph.number_of_edges(),
        "avg_degree": round(
            2 * graph.number_of_edges() / graph.number_of_nodes(), 1
        ),
    })
pd.DataFrame(rows)

C:\Users\User\AppData\Local\Temp\ipykernel_28920\2142190662.py:1: DeprecationWarning: dwave-networkx is deprecated and will be replaced by dwave-graphs in Ocean 10. Most functionality previously provided by dwave-networkx is now available as part of dwave-graphs under the 'dwave.graphs' namespace.
  import dwave_networkx as dnx


,topology,nodes,edges,avg_degree
0,pegasus16,5640,40484,14.4
1,zephyr12,4800,45864,19.1


## 대상 QUBO의 규모

embedding 난이도는 변수 수보다 **엣지 수(이차항)** 에 좌우됩니다. MS는 `q_ij`를 binary expansion하면서 한 constraint 안의 항이 늘고, 제곱 전개 시 이차항이 항 수의 제곱으로 늘어납니다.

In [7]:
from src.data_generator import CFLPInstance

rows = []
for name, formulation in TARGETS:
    instance = CFLPInstance.load(DATA_DIR / f"{name}.json")
    _, variables, edges = ES._source_graph(instance, formulation, config)
    rows.append({
        "instance": name,
        "formulation": formulation,
        "logical_variables": variables,
        "logical_edges": edges,
        "density": round(2 * edges / (variables * (variables - 1)), 4),
    })
pd.DataFrame(rows)

,instance,formulation,logical_variables,logical_edges,density
0,4x4,MS,120,2540,0.3557
1,6x6,MS,259,8701,0.2604
2,8x8,MS,413,17603,0.2069
3,15x15,MS,1439,123857,0.1197


## timeout sweep 실행 (Pegasus)

In [ ]:
# 전체 timeout sweep 실행

results = ES.run_timeout_sweep(
    targets=TARGETS,
    config=config,
    data_dir=DATA_DIR,
    timeouts=TIMEOUTS,
    seeds=SEEDS,
    tries=TRIES,
    topologies="pegasus",
    qa_dryrun=QA_DRYRUN,
)
print(f"\n{len(results)} 회 실행 완료")

[online] topology 'pegasus' solver 'Advantage_system6;graph_id=01dae5a273' 사용
  4x4 MS qpu:pegasus timeout= 300s seed=2024  성공 (  33.6s)
  4x4 MS qpu:pegasus timeout= 300s seed=2025  성공 (  23.8s)
  4x4 MS qpu:pegasus timeout= 300s seed=2026  성공 (  26.6s)
  4x4 MS qpu:pegasus timeout= 300s seed=2027  성공 (  26.7s)
  4x4 MS qpu:pegasus timeout= 300s seed=2028  성공 (  23.7s)
  4x4 MS qpu:pegasus timeout= 600s seed=2024  성공 (  32.7s)
  4x4 MS qpu:pegasus timeout= 600s seed=2025  성공 (  23.4s)
  4x4 MS qpu:pegasus timeout= 600s seed=2026  성공 (  26.6s)
  4x4 MS qpu:pegasus timeout= 600s seed=2027  성공 (  26.3s)
  4x4 MS qpu:pegasus timeout= 600s seed=2028  성공 (  23.7s)
  4x4 MS qpu:pegasus timeout=1200s seed=2024  성공 (  32.4s)
  4x4 MS qpu:pegasus timeout=1200s seed=2025  성공 (  23.4s)
  4x4 MS qpu:pegasus timeout=1200s seed=2026  성공 (  26.2s)
  4x4 MS qpu:pegasus timeout=1200s seed=2027  성공 (  26.2s)
  4x4 MS qpu:pegasus timeout=1200s seed=2028  성공 (  23.5s)
  4x4 MS qpu:pegasus timeout=2400s se

In [ ]:
# MS 6x6 / pegasus topology / timeout 2400s / 10 tries

results = ES.run_timeout_sweep(
    targets=[('6x6', 'MS')],
    config=config,
    data_dir=DATA_DIR,
    timeouts=[2400],
    seeds=SEEDS,
    tries=10,
    topologies="pegasus",
    qa_dryrun=QA_DRYRUN,
)
print(f"\n{len(results)} 회 실행 완료")

[online] topology 'pegasus' solver 'Advantage_system6;graph_id=01dae5a273' 사용
  6x6 MS qpu:pegasus timeout=2400s seed=2024  성공 ( 865.3s)
  6x6 MS qpu:pegasus timeout=2400s seed=2025  성공 ( 290.3s)
  6x6 MS qpu:pegasus timeout=2400s seed=2026  성공 (1435.6s)
  6x6 MS qpu:pegasus timeout=2400s seed=2027  성공 ( 776.8s)
  6x6 MS qpu:pegasus timeout=2400s seed=2028  성공 ( 630.6s)

5 회 실행 완료


In [9]:
# MS 8x8 / pegasus topology / timeout 2400s / 20 tries

results = ES.run_timeout_sweep(
    targets=[('8x8', 'MS')],
    config=config,
    data_dir=DATA_DIR,
    timeouts=[2400],
    seeds=SEEDS,
    tries=20,
    topologies="pegasus",
    qa_dryrun=QA_DRYRUN,
)
print(f"\n{len(results)} 회 실행 완료")

notify(f"MS 8x8 / pegasus topology / timeout 2400s / 20 tries 완료")

[online] topology 'pegasus' solver 'Advantage_system6;graph_id=01dae5a273' 사용
  8x8 MS qpu:pegasus timeout=2400s seed=2024  실패 (2418.6s)
  8x8 MS qpu:pegasus timeout=2400s seed=2025  실패 (2411.4s)
  8x8 MS qpu:pegasus timeout=2400s seed=2026  실패 (2411.5s)
  8x8 MS qpu:pegasus timeout=2400s seed=2027  실패 (2413.9s)
  8x8 MS qpu:pegasus timeout=2400s seed=2028  실패 (2409.8s)

5 회 실행 완료


In [11]:
# MS 8x8 / pegasus topology / timeout 4800s / 20 tries

results = ES.run_timeout_sweep(
    targets=[('8x8', 'MS')],
    config=config,
    data_dir=DATA_DIR,
    timeouts=[4800],
    seeds=SEEDS,
    tries=20,
    topologies="pegasus",
    qa_dryrun=QA_DRYRUN,
)
print(f"\n{len(results)} 회 실행 완료")

notify(f"MS 8x8 / pegasus topology / timeout 4800s / 20 tries 완료")

[online] topology 'pegasus' solver 'Advantage_system6;graph_id=01dae5a273' 사용
  8x8 MS qpu:pegasus timeout=4800s seed=2024  실패 (4816.4s)
  8x8 MS qpu:pegasus timeout=4800s seed=2025  실패 (4811.4s)
  8x8 MS qpu:pegasus timeout=4800s seed=2026  실패 (4814.4s)
  8x8 MS qpu:pegasus timeout=4800s seed=2027  실패 (4811.2s)
  8x8 MS qpu:pegasus timeout=4800s seed=2028  실패 (4816.3s)

5 회 실행 완료


## 결과 저장

timeout sweep과 tries sweep을 **같은 파일에 누적**합니다. notebook 21를 실행하면 이어서 쌓이고, notebook 22에서 함께 집계합니다.

In [13]:
from src.persistence import save_table

OUTPUT = PROCESSED_DIR / "embedding_study.csv"
if OUTPUT.exists():
    previous = pd.read_csv(OUTPUT)
    key = ["instance", "formulation", "topology", "topology_label",
            "sweep", "timeout", "tries", "seed"]
    merged = pd.concat([previous, results], ignore_index=True)
    merged = merged.drop_duplicates(subset=key, keep="last")
else:
    merged = results
save_table(merged, PROCESSED_DIR, "embedding_study.csv")
print(f"저장: {OUTPUT} ({len(merged)} 행)")
print(merged.groupby(["sweep", "topology_label"])["success"].agg(["count", "mean"]))

저장: C:\Users\User\Desktop\KMJ\Study\Quantum\cflp_formulation\results\processed\embedding_study.csv (190 행)
                        count      mean
sweep   topology_label                 
timeout qpu:pegasus        95  0.347368
        qpu:zephyr         95  0.210526


## 중간 판정

timeout을 늘렸을 때 성공률이 오르면 **search budget 문제**입니다. 모든 timeout에서 0%라면 budget이 아니라 topology나 formulation 쪽을 봐야 합니다.

In [9]:
pivot = results.pivot_table(
    index=["instance", "logical_variables", "logical_edges"],
    columns=["topology", "timeout"],
    values="success",
    aggfunc="mean",
)
(pivot * 100).round(0)

topology                                 pegasus                     
timeout                                     300    600    1200   2400
instance logical_variables logical_edges                             
15x15    1439              123857            0.0    0.0    0.0    0.0
4x4      120               2540            100.0  100.0  100.0  100.0
6x6      259               8701             20.0   20.0   60.0   60.0
8x8      413               17603             0.0    0.0    0.0    0.0